# LLM Fundamentals: LLM kaise kaam karta hai?
**AI Explained · Episode 16 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Is notebook mein koi asli AI nahi chalta, koi API key nahi chahiye. Ek **toy model** hai: wo sirf **pichhla ek word** dekhkar agla word guess karta hai. Asli LLM poora context dekhta hai aur bahut bade neural network ke parameters seekhta hai; ye toy sirf idea samjhane ke liye hai. Sentences hamne teaching ke liye banaye hain.

In [1]:
TEXT = [
    "thank you for your time",
    "thank you for your time",
    "thank you for your time",
    "thank you for your help",
    "thank you for your help",
    "thank you for your reply",
    "the capital of australia is canberra",
    "the oldest city of australia is sydney",
    "my favourite city of australia is sydney",
]
words = sum(len(s.split()) for s in TEXT)
print(len(TEXT), "sentences,", words, "words")

9 sentences, 50 words


In [2]:
from collections import Counter, defaultdict

def train(text):        # training = ginti
    nxt = defaultdict(Counter)
    for s in text:
        w = s.split() + ["<end>"]
        for a, b in zip(w, w[1:]):
            nxt[a][b] += 1
    return nxt

NEXT = train(TEXT)
print("your ->", dict(NEXT["your"]))

your -> {'time': 3, 'help': 2, 'reply': 1}


In [3]:
def predict(word):
    c = NEXT[word]
    total = sum(c.values())
    return {w: round(100 * n / total)
            for w, n in c.most_common()}

print(predict("your"))

{'time': 50, 'help': 33, 'reply': 17}


In [4]:
def generate(text, n=8):
    words = text.split()
    for _ in range(n):
        p = predict(words[-1])
        best = max(p, key=p.get, default=None)
        if best in (None, "<end>"):
            break
        words.append(best)   # jodo, phir agla
        print(" ".join(words))

generate("thank")

thank you
thank you for
thank you for your
thank you for your time


In [5]:
print(predict("is"))
generate("the capital of australia is")

{'sydney': 67, 'canberra': 33}
the capital of australia is sydney


In [6]:
print(predict("today"))   # data mein tha hi nahi

{}


### Toy vs asli LLM
Google ka Machine Learning Crash Course: *"A language model estimates the probability of a token or sequence of tokens occurring within a longer sequence of tokens."* Aur: *"LLMs are essentially autocomplete mechanisms that can automatically predict (complete) thousands of tokens."*

- **Training data:** asli LLM ke liye *"a phenomenal amount of training data (text)"*. Hamare paas 9 sentences hain.
- **Context:** hamara toy sirf pichhla ek word dekhta hai (n-gram jaisa). Asli LLM self-attention se poora context dekhta hai, isliye wo is sawal par shayad canberra hi bolega. Par mushkil sawalon par wahi galti hoti hai: course kehta hai *"LLMs hallucinate, meaning their predictions often contain mistakes."*
- **Bias:** *"LLMs generally share the biases of: The data they were trained on."*
- **Cutoff:** Claude ke help page par: har model ki ek knowledge cutoff hoti hai, aur us date ke baad ki baatein model ko shayad pata na hon.
- **Chat kaise bana:** pehle pre-training (agla token), phir *instruction tuning*. OpenAI ke InstructGPT paper (2022) mein humans ki ranking se bhi train kiya gaya (human feedback).

In [7]:
ANSWER = ("An LLM is trained on huge amounts "
          "of text to predict the next token. "
          "It writes an answer one token at a "
          "time, so it sounds fluent, but it "
          "can still be wrong.")
print(ANSWER)

An LLM is trained on huge amounts of text to predict the next token. It writes an answer one token at a time, so it sounds fluent, but it can still be wrong.


In [8]:
# Data badlo, phir dobara train karo
TEXT += ["the capital of australia is canberra"] * 2
NEXT = train(TEXT)
print(predict("is"))
generate("the capital of australia is")

{'canberra': 60, 'sydney': 40}
the capital of australia is canberra
